In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from worldcup.tournament import load_teams, get_groups, get_group_matches
from worldcup.simulator import GroupStageSimulator

sns.set_theme(style='whitegrid', palette='muted')

# ── Configure model here ──────────────────────────────────────────────────────
from worldcup.algorithms import UniformPredictor
predictor = UniformPredictor()
N_SIMULATIONS = 20_000
SEED = 42
# ─────────────────────────────────────────────────────────────────────────────

teams = load_teams()
groups = get_groups(teams)
flag_map  = {t.name: t.flag  for t in teams}
group_map = {t.name: t.group for t in teams}

# Uniform model

## Algorithm

The **uniform model** is the simplest possible baseline: it assigns equal probability to every outcome regardless of which two teams are playing.

$$P(\text{home win}) = P(\text{draw}) = P(\text{away win}) = \frac{1}{3}$$

This model contains **no information** about team quality. It exists to establish a floor — any algorithm worth using should beat it on accuracy once real results come in.

Because every match is treated identically, the qualification probabilities it produces reflect only the tournament structure: with 4 teams per group and 2 automatic qualifiers, each team has roughly a 50% chance of finishing top 2, plus a small additional chance via the best-3rd-place route.

### Example prediction

In [ ]:
example_home = next(t for t in teams if t.name == 'Argentina')
example_away = next(t for t in teams if t.name == 'Haiti')

p_home, p_draw, p_away = predictor.predict(example_home, example_away)

# Table display — emoji render fine in HTML
example_df = pd.DataFrame([
    {'Outcome': f'{example_home.flag} {example_home.name} win', 'Probability': p_home},
    {'Outcome': 'Draw',                                          'Probability': p_draw},
    {'Outcome': f'{example_away.flag} {example_away.name} win', 'Probability': p_away},
])
display(example_df.style.hide(axis='index')
        .format({'Probability': '{:.1%}'})
        .bar(subset='Probability', color='#90CAF9', vmin=0, vmax=1))

# Chart — plain names only (emoji don't render in matplotlib PNG output)
fig, ax = plt.subplots(figsize=(5, 2.5))
ax.bar(
    [f'{example_home.name}\nwins', 'Draw', f'{example_away.name}\nwins'],
    [p_home, p_draw, p_away],
    color=['#2196F3', '#9E9E9E', '#F44336'],
)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
ax.set_ylim(0, 0.6)
ax.set_title(f'{example_home.name}  vs  {example_away.name}', fontsize=11)
plt.tight_layout()
plt.show()

---

## Group stage qualification probabilities

Monte Carlo simulation results across all 12 groups, sorted by probability of qualifying.

In [ ]:
sim = GroupStageSimulator(predictor, n=N_SIMULATIONS, seed=SEED)
results = sim.run(groups)
df = results.to_dataframe()

df['flag']  = df['team'].map(flag_map)
df['group'] = df['team'].map(group_map)
df['Team']  = df['flag'] + ' ' + df['team']

qual_table = df[['group', 'Team', 'p_qualify', 'p_group_winner', 'p_runner_up', 'p_best_third', 'p_eliminated']].copy()
qual_table.columns = ['Group', 'Team', 'Qualify', '1st', '2nd', 'Best 3rd', 'Eliminated']

qual_table.style \
    .hide(axis='index') \
    .format({c: '{:.1%}' for c in ['Qualify', '1st', '2nd', 'Best 3rd', 'Eliminated']}) \
    .background_gradient(subset='Qualify',    cmap='RdYlGn',   vmin=0, vmax=1) \
    .background_gradient(subset='Eliminated', cmap='RdYlGn_r', vmin=0, vmax=1)

---

## Match predictions

Predicted outcome probabilities for every group stage match.

In [ ]:
match_rows = []
for group_name, group_teams in sorted(groups.items()):
    for home, away in get_group_matches(group_teams):
        p_hw, p_d, p_aw = predictor.predict(home, away)
        match_rows.append({
            'Group':      group_name,
            'Home':       f'{home.flag} {home.name}',
            'Away':       f'{away.flag} {away.name}',
            'Home win %': p_hw,
            'Draw %':     p_d,
            'Away win %': p_aw,
        })

matches_df = pd.DataFrame(match_rows)

matches_df.style \
    .hide(axis='index') \
    .format({c: '{:.1%}' for c in ['Home win %', 'Draw %', 'Away win %']}) \
    .background_gradient(subset='Home win %', cmap='Blues',  vmin=0, vmax=1) \
    .background_gradient(subset='Away win %', cmap='Reds',   vmin=0, vmax=1) \
    .background_gradient(subset='Draw %',     cmap='Greens', vmin=0, vmax=1)